# Prétraitement - Anemia Prediction

## 1. Import des bibliothèques

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

## 2. Chargement et nettoyage

In [2]:
df = pd.read_csv('anemia.csv')

print(f"Shape: {df.shape}")
print(df.isnull().sum())

Shape: (1421, 6)
Gender        0
Hemoglobin    0
MCH           0
MCHC          0
MCV           0
Result        0
dtype: int64


## 3. Traitement des outliers

On applique un capping (winsorization) sur les bornes Q1 - 1,5×IQR / Q3 + 1,5×IQR, comme pour les variables continues classiques.

In [3]:
numeric_cols = ['Hemoglobin', 'MCH', 'MCHC', 'MCV']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

print("✅ Outliers traités")

✅ Outliers traités


## 4. Split et Pipeline

In [4]:
X = df.drop('Result', axis=1)
y = df['Result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = ['Hemoglobin', 'MCH', 'MCHC', 'MCV']
# Gender est déjà encodé en 0/1 dans le dataset source : pas besoin de OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols)
], remainder='passthrough')

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Features après transformation: {X_train_processed.shape[1]}")

Features après transformation: 5


## 5. Sauvegarde

In [5]:
joblib.dump(preprocessor, 'preprocessor.pkl')

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("✅ Tous les fichiers sauvegardés")

✅ Tous les fichiers sauvegardés
